In [11]:
# run this first in a shell
# mlflow server --port 5000

In [12]:
import pandas as pd
import os
import json
import mlflow
from pathlib import Path
import numpy as np

# 1. Connect to the MLflow server
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Syria_Crop_Yield_Baseline")

def find_project_root(start: Path = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in (".git", ".dvc", "pyproject.toml", "src")):
            return parent
    raise RuntimeError(f"Project root not found from {start}")

# 2. Load the data splits you created earlier
PROJECT_ROOT  = find_project_root()
INTERIM = PROJECT_ROOT / "data" / "interim"

In [13]:
train = pd.read_csv(INTERIM / "train.csv")
val = pd.read_csv(INTERIM / "val.csv")
test = pd.read_csv(INTERIM / "test.csv")
meta = json.loads((INTERIM / "split_metadata.json").read_text())

FEATURES = meta["feature_columns"]
TARGET = meta["target"]

# Combine train + val for final training (or use train for fitting and val for early stopping)
X_train_full = pd.concat([train[FEATURES], val[FEATURES]])
y_train_full = pd.concat([train[TARGET], val[TARGET]])

X_test = test[FEATURES]
y_test = test[TARGET]

# Also keep separate validation set for XGBoost early stopping
X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]

feature_cols = X_train.columns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def eval_metrics(actual, pred):
    """Returns a dict so callers can log all metrics via .items()."""
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2   = r2_score(actual, pred)
    mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}

print(f"Crops: [{test.crop.unique()}]")

Crops: [['Apples' 'Barley' 'Chick peas, dry' 'Grapes' 'Lentils, dry'
 'Maize (corn)' 'Oats' 'Olives' 'Potatoes' 'Sesame seed' 'Sorghum'
 'Sugar beet' 'Sunflower seed' 'Tomatoes' 'Wheat']]


In [14]:
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, KFold

dt_params = {
    "max_depth"        : 6,
    "min_samples_leaf" : 3,
    "random_state"     : 42,
}

xgb_params = {
    "objective"        : "reg:squarederror",
    "eval_metric"      : "rmse",
    "max_depth"        : 3,
    "learning_rate"    : 0.1,
    "n_estimators"     : 200,
    "subsample"        : 0.9,
    "colsample_bytree" : 0.9,
    "tree_method"      : "hist",
    "random_state"     : 42,
}

# base model for each crop

## wheat

In [15]:
CROP = "Wheat"

In [16]:
train_crop = train[train["crop"] == CROP].reset_index(drop=True)
val_crop   = val[val["crop"]     == CROP].reset_index(drop=True)
test_crop  = test[test["crop"]   == CROP].reset_index(drop=True)

print(f"Crop          : {CROP}")
print(f"train_crop    : {len(train_crop):>5} rows  years {train_crop['year'].min()}–{train_crop['year'].max()}")
print(f"val_crop      : {len(val_crop):>5} rows  years {val_crop['year'].min()}–{val_crop['year'].max()}")
print(f"test_crop     : {len(test_crop):>5} rows  years {test_crop['year'].min()}–{test_crop['year'].max()}")
print(f"Features      : {FEATURES}")
print(f"Target        : {TARGET}")

X_train, y_train = train_crop[FEATURES], train_crop[TARGET]
X_val,   y_val   = val_crop[FEATURES],   val_crop[TARGET]
X_test,  y_test  = test_crop[FEATURES],  test_crop[TARGET]

Crop          : Wheat
train_crop    :    25 rows  years 1991–2015
val_crop      :     4 rows  years 2016–2019
test_crop     :     5 rows  years 2020–2024
Features      : ['t2m_mean', 't2m_max_extreme', 't2m_min_extreme', 'precip_total', 'rh2m_mean', 'solar_mean', 'ws2m_mean']
Target        : yield_hg_ha


### DTR

In [17]:
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    dt = DecisionTreeRegressor(**dt_params)
    dt.fit(X_train, y_train)

    metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = dt.predict(X)
        actual = y.values
        mlflow.log_metric(f"{split_name}_rmse", np.sqrt(mean_squared_error(actual, pred)))
        mlflow.log_metric(f"{split_name}_mae",  mean_absolute_error(actual, pred))
        mlflow.log_metric(f"{split_name}_r2",   r2_score(actual, pred))
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100
        mlflow.log_metric(f"{split_name}_mape", mape)

    mlflow.log_params(dt_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.set_tag("crop", CROP)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    mlflow.sklearn.log_model(
        sk_model=dt,
        artifact_path="decision_tree_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

    print(f"DecisionTree_{CROP}  run_id={run.info.run_id}")
    for k, v in metrics.items():
        print(f"  {k:<14} {v:.4f}")

2026/11/15 07:34:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


DecisionTree_Wheat  run_id=cb75b083f42e491e89e2c01f52b6dfad
🏃 View run DecisionTree_Wheat at: http://localhost:5000/#/experiments/1/runs/cb75b083f42e491e89e2c01f52b6dfad
🧪 View experiment at: http://localhost:5000/#/experiments/1


#### Grid Search + KFold CV

In [18]:
# ── 1. Grid definition ────────────────────────────────────────────
param_grid = {
    "max_depth":        [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "min_samples_split":[2, 4, 6],
    "ccp_alpha":        [0.0, 0.001, 0.01, 0.1],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_dt = DecisionTreeRegressor(random_state=42)
grid = GridSearchCV(
    base_dt,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_      # negate back to positive

# Log every grid trial as an MLflow run — optional but very useful
with mlflow.start_run(run_name=f"GridSearch_DecisionTree_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    # Log every trial as a child table row
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_dt = DecisionTreeRegressor(random_state=42, **best_params)
final_dt.fit(X_train_full, y_train_full)

# ── 4. Metrics helper ─────────────────────────────────────────────
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    # sMAPE — bounded in [0, 200], safe near zero
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "DecisionTree")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",       len(X_train))
    mlflow.log_metric("n_val",         len(X_val))
    mlflow.log_metric("n_test",        len(X_test))
    mlflow.log_metric("n_train_full",  len(X_train_full))
    mlflow.log_metric("cv_folds",      n_splits)
    mlflow.log_metric("best_cv_rmse",  best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train,      y_train),
        "val":   (X_val,        y_val),
        "test":  (X_test,       y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_dt.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        # Relative error ratios (unit-free, cross-crop comparable)
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    try:
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  DecisionTree_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_dt.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Top 10 feature importances:")
    for feat, imp in importances.head(10).items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics ────────────────────────────────────────
    print(f"\n  Tree depth (actual):   {final_dt.get_depth()}")
    print(f"  Tree leaves (actual):  {final_dt.get_n_leaves()}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(
            f"runs:/{run_id}/decision_tree_{CROP.lower().replace(' ', '_')}"
        )
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

d:\2026\ftl-syria-crop-yield\.venv\lib\site-packages\joblib\externals\loky\process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🏃 View run GridSearch_DecisionTree_Wheat at: http://localhost:5000/#/experiments/1/runs/0a397cc7e1154e29803c7dda56cede6e
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:35:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  DecisionTree_Wheat   run_id = 3e293e06f0574b069fbd31974b4576d1
  Best params: {'ccp_alpha': 0.0, 'max_depth': 2, 'min_samples_leaf': 2, 'min_samples_split': 2}
  CV RMSE (5-fold on train): 413.39

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       7,086.85       6,935.60 -304.848   119.78
  val       4       8,797.53       8,730.42 -414.984   146.06
  test      5       8,137.35       8,126.19 -363.976   139.64

  Top 10 feature importances:
    solar_mean              0.594  █████████████████████████████
    t2m_mean                0.406  ████████████████████
    t2m_max_extreme         0.000  
    t2m_min_extreme         0.000  
    precip_total            0.000  
    rh2m_mean               0.000  
    ws2m_mean               0.000  

  Tree depth (actual):   2
  Tree leaves (actual):  4
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run DecisionTree_Wheat at: http://localhost:5000/#/experiments/1/runs/3e293e06f0574b069fbd31974b4576d1
🧪 View experiment at: http://localhost:5000/#/experiments/1


### XGB

In [19]:


with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    # early_stopping_rounds lives in the constructor in XGBoost 2.0+
    xgb_model = XGBRegressor(
        early_stopping_rounds=min(20, max(5, len(val_crop))),
        **xgb_params,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = xgb_model.predict(X)
        actual = y.values
        mean_actual = np.mean(actual)

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100

        mlflow.log_metric(f"{split_name}_rmse", rmse)
        mlflow.log_metric(f"{split_name}_mae",  mae)
        mlflow.log_metric(f"{split_name}_r2",   r2)
        mlflow.log_metric(f"{split_name}_mape", mape)

        # Relative-error ratios
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", rmse / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  mae  / mean_actual)

    mlflow.log_params(xgb_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.log_metric("best_iteration", xgb_model.best_iteration)
    mlflow.set_tag("crop", CROP)

    mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="xgboost_model",
    )

    print(f"XGBoost_{CROP}  run_id={run.info.run_id}  best_iter={xgb_model.best_iteration}")

2026/11/15 07:36:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost_Wheat  run_id=a3e1693fe6df4cc88358440da9e02878  best_iter=12
🏃 View run XGBoost_Wheat at: http://localhost:5000/#/experiments/1/runs/a3e1693fe6df4cc88358440da9e02878
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [20]:
# ══════════════════════════════════════════════════════════════════
# XGBoostRegressor — GridSearchCV + KFold, per crop
# ══════════════════════════════════════════════════════════════════

# ── 1. Grid definition (conservative for ~20 samples/crop) ────────
param_grid = {
    "n_estimators":     [50, 100, 200],
    "max_depth":        [2, 3, 4],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,          # small data; parallel overhead not worth it
    tree_method="hist",  # faster, memory-efficient
    verbosity=0,
)

grid = GridSearchCV(
    base_xgb,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_

# Log every grid trial as a nested MLflow run
with mlflow.start_run(run_name=f"GridSearch_XGBoost_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_XGBoost_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist",
    verbosity=0,
    **best_params,
)
final_xgb.fit(X_train_full, y_train_full)

# ── 4. Metrics helper (same as DTR; redefine if not in scope) ────
# If eval_metrics is already defined from the DTR cell, skip this block.
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "XGBoost")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",      len(X_train))
    mlflow.log_metric("n_val",        len(X_val))
    mlflow.log_metric("n_test",       len(X_test))
    mlflow.log_metric("n_train_full", len(X_train_full))
    mlflow.log_metric("cv_folds",     n_splits)
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_xgb.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    artifact_name = f"xgboost_{CROP.lower().replace(' ', '_')}"
    try:
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  XGBoost_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_xgb.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Feature importances ({len(importances)} features, sorted):")
    for feat, imp in importances.items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics (XGBoost: n_estimators is depth-of-ensemble) ──
    print(f"\n  Number of trees:       {final_xgb.n_estimators}")
    print(f"  Max depth per tree:    {final_xgb.max_depth}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/{artifact_name}")
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_XGBoost_Wheat at: http://localhost:5000/#/experiments/1/runs/b37633235ac84cbab2eaaccc3faaebf6
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:38:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/15 07:38:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  [WARN] skops log failed (The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'].); retrying with cloudpickle


2026/11/15 07:38:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  XGBoost_Wheat   run_id = 6fc02695fb224b208d2f13653b56c217
  Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 1.0}
  CV RMSE (5-fold on train): 355.17

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       7,024.09       6,962.74 -299.455   120.81
  val       4       8,503.13       8,465.15 -387.610   145.10
  test      5       8,060.00       8,042.88 -357.070   139.15

  Feature importances (7 features, sorted):
    solar_mean              0.475  ███████████████████████
    t2m_mean                0.225  ███████████
    t2m_min_extreme         0.142  ███████
    ws2m_mean               0.089  ████
    t2m_max_extreme         0.070  ███
    precip_total            0.000  
    rh2m_mean               0.000  

  Number of trees:       100
  Max depth per tree:    3
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run XGBoost_Wheat at: http://localhost:5000/#/experiments/1/runs/6fc02695fb224b208d2f13653b56c217
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Maize (corn)

In [21]:
CROP = "Maize (corn)"

In [22]:
train_crop = train[train["crop"] == CROP].reset_index(drop=True)
val_crop   = val[val["crop"]     == CROP].reset_index(drop=True)
test_crop  = test[test["crop"]   == CROP].reset_index(drop=True)

print(f"Crop          : {CROP}")
print(f"train_crop    : {len(train_crop):>5} rows  years {train_crop['year'].min()}–{train_crop['year'].max()}")
print(f"val_crop      : {len(val_crop):>5} rows  years {val_crop['year'].min()}–{val_crop['year'].max()}")
print(f"test_crop     : {len(test_crop):>5} rows  years {test_crop['year'].min()}–{test_crop['year'].max()}")
print(f"Features      : {FEATURES}")
print(f"Target        : {TARGET}")

X_train, y_train = train_crop[FEATURES], train_crop[TARGET]
X_val,   y_val   = val_crop[FEATURES],   val_crop[TARGET]
X_test,  y_test  = test_crop[FEATURES],  test_crop[TARGET]

Crop          : Maize (corn)
train_crop    :    25 rows  years 1991–2015
val_crop      :     4 rows  years 2016–2019
test_crop     :     5 rows  years 2020–2024
Features      : ['t2m_mean', 't2m_max_extreme', 't2m_min_extreme', 'precip_total', 'rh2m_mean', 'solar_mean', 'ws2m_mean']
Target        : yield_hg_ha


### DTR

In [23]:
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    dt = DecisionTreeRegressor(**dt_params)
    dt.fit(X_train, y_train)

    metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = dt.predict(X)
        actual = y.values
        mlflow.log_metric(f"{split_name}_rmse", np.sqrt(mean_squared_error(actual, pred)))
        mlflow.log_metric(f"{split_name}_mae",  mean_absolute_error(actual, pred))
        mlflow.log_metric(f"{split_name}_r2",   r2_score(actual, pred))
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100
        mlflow.log_metric(f"{split_name}_mape", mape)

    mlflow.log_params(dt_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.set_tag("crop", CROP)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    mlflow.sklearn.log_model(
        sk_model=dt,
        artifact_path="decision_tree_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

    print(f"DecisionTree_{CROP}  run_id={run.info.run_id}")
    for k, v in metrics.items():
        print(f"  {k:<14} {v:.4f}")

2026/11/15 07:42:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


DecisionTree_Maize (corn)  run_id=73c13deeaa4c43d591a81185c9f81867
🏃 View run DecisionTree_Maize (corn) at: http://localhost:5000/#/experiments/1/runs/73c13deeaa4c43d591a81185c9f81867
🧪 View experiment at: http://localhost:5000/#/experiments/1


#### Grid Search + KFold CV

In [24]:
# ── 1. Grid definition ────────────────────────────────────────────
param_grid = {
    "max_depth":        [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "min_samples_split":[2, 4, 6],
    "ccp_alpha":        [0.0, 0.001, 0.01, 0.1],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_dt = DecisionTreeRegressor(random_state=42)
grid = GridSearchCV(
    base_dt,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_      # negate back to positive

# Log every grid trial as an MLflow run — optional but very useful
with mlflow.start_run(run_name=f"GridSearch_DecisionTree_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    # Log every trial as a child table row
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_dt = DecisionTreeRegressor(random_state=42, **best_params)
final_dt.fit(X_train_full, y_train_full)

# ── 4. Metrics helper ─────────────────────────────────────────────
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    # sMAPE — bounded in [0, 200], safe near zero
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "DecisionTree")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",       len(X_train))
    mlflow.log_metric("n_val",         len(X_val))
    mlflow.log_metric("n_test",        len(X_test))
    mlflow.log_metric("n_train_full",  len(X_train_full))
    mlflow.log_metric("cv_folds",      n_splits)
    mlflow.log_metric("best_cv_rmse",  best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train,      y_train),
        "val":   (X_val,        y_val),
        "test":  (X_test,       y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_dt.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        # Relative error ratios (unit-free, cross-crop comparable)
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    try:
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  DecisionTree_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_dt.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Top 10 feature importances:")
    for feat, imp in importances.head(10).items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics ────────────────────────────────────────
    print(f"\n  Tree depth (actual):   {final_dt.get_depth()}")
    print(f"  Tree leaves (actual):  {final_dt.get_n_leaves()}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(
            f"runs:/{run_id}/decision_tree_{CROP.lower().replace(' ', '_')}"
        )
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_DecisionTree_Maize (corn) at: http://localhost:5000/#/experiments/1/runs/6b0a368ed98f44d3bf93d750257eab0c
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:42:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  DecisionTree_Maize (corn)   run_id = 1e83119ef37343b18f5667c53b0b7f3f
  Best params: {'ccp_alpha': 0.0, 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 6}
  CV RMSE (5-fold on train): 653.76

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       5,526.71       5,245.24  -70.647    79.19
  val       4       7,004.24       6,967.73 -296.182    95.01
  test      5       4,119.11       4,018.42  -22.054    52.46

  Top 10 feature importances:
    solar_mean              0.535  ██████████████████████████
    t2m_mean                0.321  ████████████████
    rh2m_mean               0.082  ████
    t2m_min_extreme         0.058  ██
    ws2m_mean               0.004  
    t2m_max_extreme         0.000  
    precip_total            0.000  

  Tree depth (actual):   5
  Tree leaves (actual):  15
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run DecisionTree_Maize (corn) at: http://localhost:5000/#/experiments/1/runs/1e83119ef37343b18f5667c53b0b7f3f
🧪 View experiment at: http://localhost:5000/#/experiments/1


### XGB

In [25]:


with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    # early_stopping_rounds lives in the constructor in XGBoost 2.0+
    xgb_model = XGBRegressor(
        early_stopping_rounds=min(20, max(5, len(val_crop))),
        **xgb_params,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = xgb_model.predict(X)
        actual = y.values
        mean_actual = np.mean(actual)

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100

        mlflow.log_metric(f"{split_name}_rmse", rmse)
        mlflow.log_metric(f"{split_name}_mae",  mae)
        mlflow.log_metric(f"{split_name}_r2",   r2)
        mlflow.log_metric(f"{split_name}_mape", mape)

        # Relative-error ratios
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", rmse / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  mae  / mean_actual)

    mlflow.log_params(xgb_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.log_metric("best_iteration", xgb_model.best_iteration)
    mlflow.set_tag("crop", CROP)

    mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="xgboost_model",
    )

    print(f"XGBoost_{CROP}  run_id={run.info.run_id}  best_iter={xgb_model.best_iteration}")

2026/11/15 07:43:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost_Maize (corn)  run_id=05aaab462c5e48d199c5e1a44aa094d8  best_iter=0
🏃 View run XGBoost_Maize (corn) at: http://localhost:5000/#/experiments/1/runs/05aaab462c5e48d199c5e1a44aa094d8
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [26]:
# ══════════════════════════════════════════════════════════════════
# XGBoostRegressor — GridSearchCV + KFold, per crop
# ══════════════════════════════════════════════════════════════════

# ── 1. Grid definition (conservative for ~20 samples/crop) ────────
param_grid = {
    "n_estimators":     [50, 100, 200],
    "max_depth":        [2, 3, 4],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,          # small data; parallel overhead not worth it
    tree_method="hist",  # faster, memory-efficient
    verbosity=0,
)

grid = GridSearchCV(
    base_xgb,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_

# Log every grid trial as a nested MLflow run
with mlflow.start_run(run_name=f"GridSearch_XGBoost_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_XGBoost_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist",
    verbosity=0,
    **best_params,
)
final_xgb.fit(X_train_full, y_train_full)

# ── 4. Metrics helper (same as DTR; redefine if not in scope) ────
# If eval_metrics is already defined from the DTR cell, skip this block.
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "XGBoost")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",      len(X_train))
    mlflow.log_metric("n_val",        len(X_val))
    mlflow.log_metric("n_test",       len(X_test))
    mlflow.log_metric("n_train_full", len(X_train_full))
    mlflow.log_metric("cv_folds",     n_splits)
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_xgb.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    artifact_name = f"xgboost_{CROP.lower().replace(' ', '_')}"
    try:
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  XGBoost_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_xgb.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Feature importances ({len(importances)} features, sorted):")
    for feat, imp in importances.items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics (XGBoost: n_estimators is depth-of-ensemble) ──
    print(f"\n  Number of trees:       {final_xgb.n_estimators}")
    print(f"  Max depth per tree:    {final_xgb.max_depth}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/{artifact_name}")
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_XGBoost_Maize (corn) at: http://localhost:5000/#/experiments/1/runs/65e0c22f78b64ccd9311698a156aa738
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:45:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/15 07:45:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  [WARN] skops log failed (The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'].); retrying with cloudpickle


2026/11/15 07:45:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  XGBoost_Maize (corn)   run_id = 2f9f1bc533fe4716bb653613e8a23e16
  Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 50, 'subsample': 1.0}
  CV RMSE (5-fold on train): 591.55

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       5,461.19       5,387.86  -68.958    82.13
  val       4       5,864.81       5,842.26 -207.357    86.43
  test      5       3,920.56       3,825.34  -19.885    50.90

  Feature importances (7 features, sorted):
    solar_mean              0.568  ████████████████████████████
    t2m_mean                0.222  ███████████
    ws2m_mean               0.121  ██████
    t2m_max_extreme         0.077  ███
    t2m_min_extreme         0.013  
    precip_total            0.000  
    rh2m_mean               0.000  

  Number of trees:       50
  Max depth per tree:    2
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run XGBoost_Maize (corn) at: http://localhost:5000/#/experiments/1/runs/2f9f1bc533fe4716bb653613e8a23e16
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Olives

In [27]:
CROP = "Olives"

In [28]:
train_crop = train[train["crop"] == CROP].reset_index(drop=True)
val_crop   = val[val["crop"]     == CROP].reset_index(drop=True)
test_crop  = test[test["crop"]   == CROP].reset_index(drop=True)

print(f"Crop          : {CROP}")
print(f"train_crop    : {len(train_crop):>5} rows  years {train_crop['year'].min()}–{train_crop['year'].max()}")
print(f"val_crop      : {len(val_crop):>5} rows  years {val_crop['year'].min()}–{val_crop['year'].max()}")
print(f"test_crop     : {len(test_crop):>5} rows  years {test_crop['year'].min()}–{test_crop['year'].max()}")
print(f"Features      : {FEATURES}")
print(f"Target        : {TARGET}")

X_train, y_train = train_crop[FEATURES], train_crop[TARGET]
X_val,   y_val   = val_crop[FEATURES],   val_crop[TARGET]
X_test,  y_test  = test_crop[FEATURES],  test_crop[TARGET]

Crop          : Olives
train_crop    :    25 rows  years 1991–2015
val_crop      :     4 rows  years 2016–2019
test_crop     :     5 rows  years 2020–2024
Features      : ['t2m_mean', 't2m_max_extreme', 't2m_min_extreme', 'precip_total', 'rh2m_mean', 'solar_mean', 'ws2m_mean']
Target        : yield_hg_ha


### DTR

In [29]:
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    dt = DecisionTreeRegressor(**dt_params)
    dt.fit(X_train, y_train)

    metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = dt.predict(X)
        actual = y.values
        mlflow.log_metric(f"{split_name}_rmse", np.sqrt(mean_squared_error(actual, pred)))
        mlflow.log_metric(f"{split_name}_mae",  mean_absolute_error(actual, pred))
        mlflow.log_metric(f"{split_name}_r2",   r2_score(actual, pred))
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100
        mlflow.log_metric(f"{split_name}_mape", mape)

    mlflow.log_params(dt_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.set_tag("crop", CROP)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    mlflow.sklearn.log_model(
        sk_model=dt,
        artifact_path="decision_tree_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

    print(f"DecisionTree_{CROP}  run_id={run.info.run_id}")
    for k, v in metrics.items():
        print(f"  {k:<14} {v:.4f}")

2026/11/15 07:47:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


DecisionTree_Olives  run_id=64cf37dbae0a4114af6a482753eef14b
🏃 View run DecisionTree_Olives at: http://localhost:5000/#/experiments/1/runs/64cf37dbae0a4114af6a482753eef14b
🧪 View experiment at: http://localhost:5000/#/experiments/1


#### Grid Search + KFold CV

In [30]:
# ── 1. Grid definition ────────────────────────────────────────────
param_grid = {
    "max_depth":        [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "min_samples_split":[2, 4, 6],
    "ccp_alpha":        [0.0, 0.001, 0.01, 0.1],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_dt = DecisionTreeRegressor(random_state=42)
grid = GridSearchCV(
    base_dt,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_      # negate back to positive

# Log every grid trial as an MLflow run — optional but very useful
with mlflow.start_run(run_name=f"GridSearch_DecisionTree_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    # Log every trial as a child table row
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_dt = DecisionTreeRegressor(random_state=42, **best_params)
final_dt.fit(X_train_full, y_train_full)

# ── 4. Metrics helper ─────────────────────────────────────────────
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    # sMAPE — bounded in [0, 200], safe near zero
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "DecisionTree")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",       len(X_train))
    mlflow.log_metric("n_val",         len(X_val))
    mlflow.log_metric("n_test",        len(X_test))
    mlflow.log_metric("n_train_full",  len(X_train_full))
    mlflow.log_metric("cv_folds",      n_splits)
    mlflow.log_metric("best_cv_rmse",  best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train,      y_train),
        "val":   (X_val,        y_val),
        "test":  (X_test,       y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_dt.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        # Relative error ratios (unit-free, cross-crop comparable)
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    try:
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  DecisionTree_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_dt.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Top 10 feature importances:")
    for feat, imp in importances.head(10).items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics ────────────────────────────────────────
    print(f"\n  Tree depth (actual):   {final_dt.get_depth()}")
    print(f"  Tree leaves (actual):  {final_dt.get_n_leaves()}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(
            f"runs:/{run_id}/decision_tree_{CROP.lower().replace(' ', '_')}"
        )
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_DecisionTree_Olives at: http://localhost:5000/#/experiments/1/runs/75672d7eb6df4886aee6d60b612f5197
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:48:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  DecisionTree_Olives   run_id = b16e5d06b62e4db0b929e9e14d24709d
  Best params: {'ccp_alpha': 0.0, 'max_depth': 2, 'min_samples_leaf': 5, 'min_samples_split': 2}
  CV RMSE (5-fold on train): 527.96

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       8,048.89       7,936.16 -387.247   151.37
  val       4       9,280.28       9,242.59 -5085.752   161.51
  test      5       8,785.80       8,783.29 -1755.510   159.55

  Top 10 feature importances:
    solar_mean              0.594  █████████████████████████████
    t2m_mean                0.406  ████████████████████
    t2m_max_extreme         0.000  
    t2m_min_extreme         0.000  
    precip_total            0.000  
    rh2m_mean               0.000  
    ws2m_mean               0.000  

  Tree depth (actual):   2
  Tree leaves (actual):  4
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run DecisionTree_Olives at: http://localhost:5000/#/experiments/1/runs/b16e5d06b62e4db0b929e9e14d24709d
🧪 View experiment at: http://localhost:5000/#/experiments/1


### XGB

In [31]:


with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    # early_stopping_rounds lives in the constructor in XGBoost 2.0+
    xgb_model = XGBRegressor(
        early_stopping_rounds=min(20, max(5, len(val_crop))),
        **xgb_params,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = xgb_model.predict(X)
        actual = y.values
        mean_actual = np.mean(actual)

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100

        mlflow.log_metric(f"{split_name}_rmse", rmse)
        mlflow.log_metric(f"{split_name}_mae",  mae)
        mlflow.log_metric(f"{split_name}_r2",   r2)
        mlflow.log_metric(f"{split_name}_mape", mape)

        # Relative-error ratios
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", rmse / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  mae  / mean_actual)

    mlflow.log_params(xgb_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.log_metric("best_iteration", xgb_model.best_iteration)
    mlflow.set_tag("crop", CROP)

    mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="xgboost_model",
    )

    print(f"XGBoost_{CROP}  run_id={run.info.run_id}  best_iter={xgb_model.best_iteration}")

2026/11/15 07:50:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost_Olives  run_id=a02ec540fb8c407a90afabc94dcbd1cd  best_iter=4
🏃 View run XGBoost_Olives at: http://localhost:5000/#/experiments/1/runs/a02ec540fb8c407a90afabc94dcbd1cd
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [32]:
# ══════════════════════════════════════════════════════════════════
# XGBoostRegressor — GridSearchCV + KFold, per crop
# ══════════════════════════════════════════════════════════════════

# ── 1. Grid definition (conservative for ~20 samples/crop) ────────
param_grid = {
    "n_estimators":     [50, 100, 200],
    "max_depth":        [2, 3, 4],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,          # small data; parallel overhead not worth it
    tree_method="hist",  # faster, memory-efficient
    verbosity=0,
)

grid = GridSearchCV(
    base_xgb,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_

# Log every grid trial as a nested MLflow run
with mlflow.start_run(run_name=f"GridSearch_XGBoost_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_XGBoost_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist",
    verbosity=0,
    **best_params,
)
final_xgb.fit(X_train_full, y_train_full)

# ── 4. Metrics helper (same as DTR; redefine if not in scope) ────
# If eval_metrics is already defined from the DTR cell, skip this block.
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "XGBoost")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",      len(X_train))
    mlflow.log_metric("n_val",        len(X_val))
    mlflow.log_metric("n_test",       len(X_test))
    mlflow.log_metric("n_train_full", len(X_train_full))
    mlflow.log_metric("cv_folds",     n_splits)
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_xgb.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    artifact_name = f"xgboost_{CROP.lower().replace(' ', '_')}"
    try:
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  XGBoost_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_xgb.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Feature importances ({len(importances)} features, sorted):")
    for feat, imp in importances.items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics (XGBoost: n_estimators is depth-of-ensemble) ──
    print(f"\n  Number of trees:       {final_xgb.n_estimators}")
    print(f"  Max depth per tree:    {final_xgb.max_depth}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/{artifact_name}")
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_XGBoost_Olives at: http://localhost:5000/#/experiments/1/runs/190e48a01938459b8aaabfdcae13417c
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:55:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/15 07:55:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  [WARN] skops log failed (The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'].); retrying with cloudpickle


2026/11/15 07:55:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  XGBoost_Olives   run_id = 3faf75ec34164e649c22d18e8fc22ca0
  Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 50, 'subsample': 0.8}
  CV RMSE (5-fold on train): 407.47

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       7,983.45       7,968.42 -380.960   151.97
  val       4       8,822.09       8,816.68 -4595.865   160.27
  test      5       8,627.61       8,627.15 -1692.827   159.03

  Feature importances (7 features, sorted):
    rh2m_mean               0.171  ████████
    solar_mean              0.168  ████████
    precip_total            0.154  ███████
    ws2m_mean               0.138  ██████
    t2m_max_extreme         0.125  ██████
    t2m_min_extreme         0.123  ██████
    t2m_mean                0.121  ██████

  Number of trees:       50
  Max depth per tree:    3
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run XGBoost_Olives at: http://localhost:5000/#/experiments/1/runs/3faf75ec34164e649c22d18e8fc22ca0
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Potatoes

In [33]:
CROP = "Potatoes"

In [34]:
train_crop = train[train["crop"] == CROP].reset_index(drop=True)
val_crop   = val[val["crop"]     == CROP].reset_index(drop=True)
test_crop  = test[test["crop"]   == CROP].reset_index(drop=True)

print(f"Crop          : {CROP}")
print(f"train_crop    : {len(train_crop):>5} rows  years {train_crop['year'].min()}–{train_crop['year'].max()}")
print(f"val_crop      : {len(val_crop):>5} rows  years {val_crop['year'].min()}–{val_crop['year'].max()}")
print(f"test_crop     : {len(test_crop):>5} rows  years {test_crop['year'].min()}–{test_crop['year'].max()}")
print(f"Features      : {FEATURES}")
print(f"Target        : {TARGET}")

X_train, y_train = train_crop[FEATURES], train_crop[TARGET]
X_val,   y_val   = val_crop[FEATURES],   val_crop[TARGET]
X_test,  y_test  = test_crop[FEATURES],  test_crop[TARGET]

Crop          : Potatoes
train_crop    :    25 rows  years 1991–2015
val_crop      :     4 rows  years 2016–2019
test_crop     :     5 rows  years 2020–2024
Features      : ['t2m_mean', 't2m_max_extreme', 't2m_min_extreme', 'precip_total', 'rh2m_mean', 'solar_mean', 'ws2m_mean']
Target        : yield_hg_ha


### DTR

In [35]:
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    dt = DecisionTreeRegressor(**dt_params)
    dt.fit(X_train, y_train)

    metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = dt.predict(X)
        actual = y.values
        mlflow.log_metric(f"{split_name}_rmse", np.sqrt(mean_squared_error(actual, pred)))
        mlflow.log_metric(f"{split_name}_mae",  mean_absolute_error(actual, pred))
        mlflow.log_metric(f"{split_name}_r2",   r2_score(actual, pred))
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100
        mlflow.log_metric(f"{split_name}_mape", mape)

    mlflow.log_params(dt_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.set_tag("crop", CROP)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    mlflow.sklearn.log_model(
        sk_model=dt,
        artifact_path="decision_tree_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

    print(f"DecisionTree_{CROP}  run_id={run.info.run_id}")
    for k, v in metrics.items():
        print(f"  {k:<14} {v:.4f}")

2026/11/15 07:55:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


DecisionTree_Potatoes  run_id=c7e38835f72f462792d58d1dbabd57f6
🏃 View run DecisionTree_Potatoes at: http://localhost:5000/#/experiments/1/runs/c7e38835f72f462792d58d1dbabd57f6
🧪 View experiment at: http://localhost:5000/#/experiments/1


#### Grid Search + KFold CV

In [36]:
# ── 1. Grid definition ────────────────────────────────────────────
param_grid = {
    "max_depth":        [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "min_samples_split":[2, 4, 6],
    "ccp_alpha":        [0.0, 0.001, 0.01, 0.1],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_dt = DecisionTreeRegressor(random_state=42)
grid = GridSearchCV(
    base_dt,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_      # negate back to positive

# Log every grid trial as an MLflow run — optional but very useful
with mlflow.start_run(run_name=f"GridSearch_DecisionTree_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    # Log every trial as a child table row
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_dt = DecisionTreeRegressor(random_state=42, **best_params)
final_dt.fit(X_train_full, y_train_full)

# ── 4. Metrics helper ─────────────────────────────────────────────
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    # sMAPE — bounded in [0, 200], safe near zero
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "DecisionTree")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",       len(X_train))
    mlflow.log_metric("n_val",         len(X_val))
    mlflow.log_metric("n_test",        len(X_test))
    mlflow.log_metric("n_train_full",  len(X_train_full))
    mlflow.log_metric("cv_folds",      n_splits)
    mlflow.log_metric("best_cv_rmse",  best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train,      y_train),
        "val":   (X_val,        y_val),
        "test":  (X_test,       y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_dt.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        # Relative error ratios (unit-free, cross-crop comparable)
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    try:
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  DecisionTree_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_dt.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Top 10 feature importances:")
    for feat, imp in importances.head(10).items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics ────────────────────────────────────────
    print(f"\n  Tree depth (actual):   {final_dt.get_depth()}")
    print(f"  Tree leaves (actual):  {final_dt.get_n_leaves()}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(
            f"runs:/{run_id}/decision_tree_{CROP.lower().replace(' ', '_')}"
        )
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_DecisionTree_Potatoes at: http://localhost:5000/#/experiments/1/runs/3c85b26114084d46a9d53be867720545
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 07:58:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  DecisionTree_Potatoes   run_id = acafdf5061354769884633c094818ee2
  Best params: {'ccp_alpha': 0.0, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 6}
  CV RMSE (5-fold on train): 1,897.96

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25      10,560.94      10,458.54  -35.932    72.89
  val       4      13,882.13      13,835.51 -111.997    80.21
  test      5      13,760.95      13,736.39 -279.386    81.84

  Top 10 feature importances:
    solar_mean              0.594  █████████████████████████████
    t2m_mean                0.406  ████████████████████
    t2m_max_extreme         0.000  
    t2m_min_extreme         0.000  
    precip_total            0.000  
    rh2m_mean               0.000  
    ws2m_mean               0.000  

  Tree depth (actual):   2
  Tree leaves (actual):  4
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run DecisionTree_Potatoes at: http://localhost:5000/#/experiments/1/runs/acafdf5061354769884633c094818ee2
🧪 View experiment at: http://localhost:5000/#/experiments/1


### XGB

In [37]:


with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    # early_stopping_rounds lives in the constructor in XGBoost 2.0+
    xgb_model = XGBRegressor(
        early_stopping_rounds=min(20, max(5, len(val_crop))),
        **xgb_params,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = xgb_model.predict(X)
        actual = y.values
        mean_actual = np.mean(actual)

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100

        mlflow.log_metric(f"{split_name}_rmse", rmse)
        mlflow.log_metric(f"{split_name}_mae",  mae)
        mlflow.log_metric(f"{split_name}_r2",   r2)
        mlflow.log_metric(f"{split_name}_mape", mape)

        # Relative-error ratios
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", rmse / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  mae  / mean_actual)

    mlflow.log_params(xgb_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.log_metric("best_iteration", xgb_model.best_iteration)
    mlflow.set_tag("crop", CROP)

    mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="xgboost_model",
    )

    print(f"XGBoost_{CROP}  run_id={run.info.run_id}  best_iter={xgb_model.best_iteration}")

2026/11/15 08:01:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost_Potatoes  run_id=54fb9e83e58e40f7aa76c00b552c6957  best_iter=10
🏃 View run XGBoost_Potatoes at: http://localhost:5000/#/experiments/1/runs/54fb9e83e58e40f7aa76c00b552c6957
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [38]:
# ══════════════════════════════════════════════════════════════════
# XGBoostRegressor — GridSearchCV + KFold, per crop
# ══════════════════════════════════════════════════════════════════

# ── 1. Grid definition (conservative for ~20 samples/crop) ────────
param_grid = {
    "n_estimators":     [50, 100, 200],
    "max_depth":        [2, 3, 4],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,          # small data; parallel overhead not worth it
    tree_method="hist",  # faster, memory-efficient
    verbosity=0,
)

grid = GridSearchCV(
    base_xgb,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_

# Log every grid trial as a nested MLflow run
with mlflow.start_run(run_name=f"GridSearch_XGBoost_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_XGBoost_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist",
    verbosity=0,
    **best_params,
)
final_xgb.fit(X_train_full, y_train_full)

# ── 4. Metrics helper (same as DTR; redefine if not in scope) ────
# If eval_metrics is already defined from the DTR cell, skip this block.
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "XGBoost")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",      len(X_train))
    mlflow.log_metric("n_val",        len(X_val))
    mlflow.log_metric("n_test",       len(X_test))
    mlflow.log_metric("n_train_full", len(X_train_full))
    mlflow.log_metric("cv_folds",     n_splits)
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_xgb.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    artifact_name = f"xgboost_{CROP.lower().replace(' ', '_')}"
    try:
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  XGBoost_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_xgb.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Feature importances ({len(importances)} features, sorted):")
    for feat, imp in importances.items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics (XGBoost: n_estimators is depth-of-ensemble) ──
    print(f"\n  Number of trees:       {final_xgb.n_estimators}")
    print(f"  Max depth per tree:    {final_xgb.max_depth}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/{artifact_name}")
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_XGBoost_Potatoes at: http://localhost:5000/#/experiments/1/runs/0eeabdf4c65240aa9e510a13ecfc3b0b
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 08:06:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/15 08:06:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  [WARN] skops log failed (The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'].); retrying with cloudpickle


2026/11/15 08:06:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  XGBoost_Potatoes   run_id = 601cbd391dd545eebf61a4770751fc94
  Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 50, 'subsample': 0.8}
  CV RMSE (5-fold on train): 1,636.22

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25      10,553.97      10,436.88  -35.884    71.87
  val       4      14,239.46      14,191.17 -117.889    82.98
  test      5      13,942.12      13,920.71 -286.818    83.42

  Feature importances (7 features, sorted):
    solar_mean              0.196  █████████
    rh2m_mean               0.170  ████████
    ws2m_mean               0.155  ███████
    precip_total            0.155  ███████
    t2m_min_extreme         0.118  █████
    t2m_mean                0.110  █████
    t2m_max_extreme         0.096  ████

  Number of trees:       50
  Max depth per tree:    4
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run XGBoost_Potatoes at: http://localhost:5000/#/experiments/1/runs/601cbd391dd545eebf61a4770751fc94
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Sunflower seed

In [39]:
CROP = "Sunflower seed"

In [40]:
train_crop = train[train["crop"] == CROP].reset_index(drop=True)
val_crop   = val[val["crop"]     == CROP].reset_index(drop=True)
test_crop  = test[test["crop"]   == CROP].reset_index(drop=True)

print(f"Crop          : {CROP}")
print(f"train_crop    : {len(train_crop):>5} rows  years {train_crop['year'].min()}–{train_crop['year'].max()}")
print(f"val_crop      : {len(val_crop):>5} rows  years {val_crop['year'].min()}–{val_crop['year'].max()}")
print(f"test_crop     : {len(test_crop):>5} rows  years {test_crop['year'].min()}–{test_crop['year'].max()}")
print(f"Features      : {FEATURES}")
print(f"Target        : {TARGET}")

X_train, y_train = train_crop[FEATURES], train_crop[TARGET]
X_val,   y_val   = val_crop[FEATURES],   val_crop[TARGET]
X_test,  y_test  = test_crop[FEATURES],  test_crop[TARGET]

Crop          : Sunflower seed
train_crop    :    25 rows  years 1991–2015
val_crop      :     4 rows  years 2016–2019
test_crop     :     5 rows  years 2020–2024
Features      : ['t2m_mean', 't2m_max_extreme', 't2m_min_extreme', 'precip_total', 'rh2m_mean', 'solar_mean', 'ws2m_mean']
Target        : yield_hg_ha


### DTR

In [41]:
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    dt = DecisionTreeRegressor(**dt_params)
    dt.fit(X_train, y_train)

    metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = dt.predict(X)
        actual = y.values
        mlflow.log_metric(f"{split_name}_rmse", np.sqrt(mean_squared_error(actual, pred)))
        mlflow.log_metric(f"{split_name}_mae",  mean_absolute_error(actual, pred))
        mlflow.log_metric(f"{split_name}_r2",   r2_score(actual, pred))
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100
        mlflow.log_metric(f"{split_name}_mape", mape)

    mlflow.log_params(dt_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.set_tag("crop", CROP)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    mlflow.sklearn.log_model(
        sk_model=dt,
        artifact_path="decision_tree_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

    print(f"DecisionTree_{CROP}  run_id={run.info.run_id}")
    for k, v in metrics.items():
        print(f"  {k:<14} {v:.4f}")

2026/11/15 08:06:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


DecisionTree_Sunflower seed  run_id=67302a29997c4b8c915a4c474cc6e545
🏃 View run DecisionTree_Sunflower seed at: http://localhost:5000/#/experiments/1/runs/67302a29997c4b8c915a4c474cc6e545
🧪 View experiment at: http://localhost:5000/#/experiments/1


#### Grid Search + KFold CV

In [42]:
# ── 1. Grid definition ────────────────────────────────────────────
param_grid = {
    "max_depth":        [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "min_samples_split":[2, 4, 6],
    "ccp_alpha":        [0.0, 0.001, 0.01, 0.1],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_dt = DecisionTreeRegressor(random_state=42)
grid = GridSearchCV(
    base_dt,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_      # negate back to positive

# Log every grid trial as an MLflow run — optional but very useful
with mlflow.start_run(run_name=f"GridSearch_DecisionTree_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    # Log every trial as a child table row
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_dt = DecisionTreeRegressor(random_state=42, **best_params)
final_dt.fit(X_train_full, y_train_full)

# ── 4. Metrics helper ─────────────────────────────────────────────
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    # sMAPE — bounded in [0, 200], safe near zero
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"DecisionTree_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "DecisionTree")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",       len(X_train))
    mlflow.log_metric("n_val",         len(X_val))
    mlflow.log_metric("n_test",        len(X_test))
    mlflow.log_metric("n_train_full",  len(X_train_full))
    mlflow.log_metric("cv_folds",      n_splits)
    mlflow.log_metric("best_cv_rmse",  best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train,      y_train),
        "val":   (X_val,        y_val),
        "test":  (X_test,       y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_dt.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        # Relative error ratios (unit-free, cross-crop comparable)
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    try:
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_dt,
            artifact_path=f"decision_tree_{CROP.lower().replace(' ', '_')}",
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  DecisionTree_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_dt.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Top 10 feature importances:")
    for feat, imp in importances.head(10).items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics ────────────────────────────────────────
    print(f"\n  Tree depth (actual):   {final_dt.get_depth()}")
    print(f"  Tree leaves (actual):  {final_dt.get_n_leaves()}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(
            f"runs:/{run_id}/decision_tree_{CROP.lower().replace(' ', '_')}"
        )
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_DecisionTree_Sunflower seed at: http://localhost:5000/#/experiments/1/runs/09765a6149794e58ae66989716e8ef7c
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 08:07:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  DecisionTree_Sunflower seed   run_id = 9a84a755d0e74316917d7a1d66a19580
  Best params: {'ccp_alpha': 0.0, 'max_depth': 2, 'min_samples_leaf': 3, 'min_samples_split': 2}
  CV RMSE (5-fold on train): 227.94

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       7,670.66       7,539.50 -664.809   137.68
  val       4       8,770.86       8,735.89 -21102.997   146.16
  test      5       8,470.78       8,470.57 -20653.119   149.51

  Top 10 feature importances:
    solar_mean              0.594  █████████████████████████████
    t2m_mean                0.406  ████████████████████
    t2m_max_extreme         0.000  
    t2m_min_extreme         0.000  
    precip_total            0.000  
    rh2m_mean               0.000  
    ws2m_mean               0.000  

  Tree depth (actual):   2
  Tree leaves (actual):  4
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run DecisionTree_Sunflower seed at: http://localhost:5000/#/experiments/1/runs/9a84a755d0e74316917d7a1d66a19580
🧪 View experiment at: http://localhost:5000/#/experiments/1


### XGB

In [43]:


with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    # early_stopping_rounds lives in the constructor in XGBoost 2.0+
    xgb_model = XGBRegressor(
        early_stopping_rounds=min(20, max(5, len(val_crop))),
        **xgb_params,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        pred = xgb_model.predict(X)
        actual = y.values
        mean_actual = np.mean(actual)

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1, actual))) * 100

        mlflow.log_metric(f"{split_name}_rmse", rmse)
        mlflow.log_metric(f"{split_name}_mae",  mae)
        mlflow.log_metric(f"{split_name}_r2",   r2)
        mlflow.log_metric(f"{split_name}_mape", mape)

        # Relative-error ratios
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", rmse / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  mae  / mean_actual)

    mlflow.log_params(xgb_params)
    mlflow.log_metric("n_train", len(train_crop))
    mlflow.log_metric("n_val",   len(val_crop))
    mlflow.log_metric("n_test",  len(test_crop))
    mlflow.log_metric("best_iteration", xgb_model.best_iteration)
    mlflow.set_tag("crop", CROP)

    mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="xgboost_model",
    )

    print(f"XGBoost_{CROP}  run_id={run.info.run_id}  best_iter={xgb_model.best_iteration}")

2026/11/15 08:09:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost_Sunflower seed  run_id=7bac971e0051400e8ad86f01b86e00ff  best_iter=8
🏃 View run XGBoost_Sunflower seed at: http://localhost:5000/#/experiments/1/runs/7bac971e0051400e8ad86f01b86e00ff
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [44]:
# ══════════════════════════════════════════════════════════════════
# XGBoostRegressor — GridSearchCV + KFold, per crop
# ══════════════════════════════════════════════════════════════════

# ── 1. Grid definition (conservative for ~20 samples/crop) ────────
param_grid = {
    "n_estimators":     [50, 100, 200],
    "max_depth":        [2, 3, 4],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3],
}

n_splits = min(5, len(X_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# ── 2. Grid search on train (CV selects best params) ──────────────
base_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,          # small data; parallel overhead not worth it
    tree_method="hist",  # faster, memory-efficient
    verbosity=0,
)

grid = GridSearchCV(
    base_xgb,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
grid.fit(X_train, y_train)

best_params = grid.best_params_
best_cv_rmse = -grid.best_score_

# Log every grid trial as a nested MLflow run
with mlflow.start_run(run_name=f"GridSearch_XGBoost_{CROP}", nested=True):
    mlflow.log_params({"crop": CROP, "cv_folds": n_splits,
                       "n_grid_combos": len(grid.cv_results_["params"])})
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)
    for k, v in best_params.items():
        mlflow.log_param(f"best_{k}", v)
    trials = pd.DataFrame(grid.cv_results_)
    trials_dir  = os.path.join("mlruns", "grid_trials")
    os.makedirs(trials_dir, exist_ok=True)
    trials_path = os.path.join(trials_dir, f"grid_trials_XGBoost_{CROP}.csv")
    trials.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)

# ── 3. Final model — refit on train+val with best params ──────────
final_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist",
    verbosity=0,
    **best_params,
)
final_xgb.fit(X_train_full, y_train_full)

# ── 4. Metrics helper (same as DTR; redefine if not in scope) ────
# If eval_metrics is already defined from the DTR cell, skip this block.
def eval_metrics(actual, pred):
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae  = float(mean_absolute_error(actual, pred))
    r2   = float(r2_score(actual, pred))
    smape = float(np.mean(2 * np.abs(actual - pred)
                          / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100)
    return {"rmse": rmse, "mae": mae, "r2": r2, "smape": smape}

# ── 5. Main run — final model, all three splits ───────────────────
with mlflow.start_run(run_name=f"XGBoost_{CROP}") as run:
    mlflow.set_tag("crop", CROP)
    mlflow.set_tag("model_family", "XGBoost")
    mlflow.log_params(best_params)
    mlflow.log_metric("n_train",      len(X_train))
    mlflow.log_metric("n_val",        len(X_val))
    mlflow.log_metric("n_test",       len(X_test))
    mlflow.log_metric("n_train_full", len(X_train_full))
    mlflow.log_metric("cv_folds",     n_splits)
    mlflow.log_metric("best_cv_rmse", best_cv_rmse)

    all_metrics = {}
    for split_name, (X, y) in {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }.items():
        if len(y) == 0:
            continue
        pred = final_xgb.predict(X)
        actual = y.values
        m = eval_metrics(actual, pred)
        for k, v in m.items():
            mlflow.log_metric(f"{split_name}_{k}", v)
            all_metrics[f"{split_name}_{k}"] = v
        mean_actual = float(np.mean(actual))
        if mean_actual != 0:
            mlflow.log_metric(f"{split_name}_rmse_ratio", m["rmse"] / mean_actual)
            mlflow.log_metric(f"{split_name}_mae_ratio",  m["mae"]  / mean_actual)

    # Save the model (try skops, fall back to cloudpickle)
    artifact_name = f"xgboost_{CROP.lower().replace(' ', '_')}"
    try:
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    except Exception as e:
        print(f"  [WARN] skops log failed ({e}); retrying with cloudpickle")
        mlflow.sklearn.log_model(
            final_xgb,
            artifact_path=artifact_name,
            serialization_format="cloudpickle",
        )

    run_id = run.info.run_id

    # ── Print summary ────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  XGBoost_{CROP}   run_id = {run_id}")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE ({n_splits}-fold on train): {best_cv_rmse:,.2f}")
    print(f"{'='*72}")
    print(f"\n  {'split':<6} {'n':>4} {'rmse':>14} {'mae':>14} {'r2':>8} {'smape':>8}")
    print(f"  {'-'*6} {'-'*4:>4} {'-'*14:>14} {'-'*14:>14} {'-'*8:>8} {'-'*8:>8}")
    for split in ("train", "val", "test"):
        rmse = all_metrics.get(f"{split}_rmse")
        if rmse is None:
            continue
        n = {"train": len(X_train), "val": len(X_val), "test": len(X_test)}[split]
        print(f"  {split:<6} {n:>4} "
              f"{all_metrics[f'{split}_rmse']:>14,.2f} "
              f"{all_metrics[f'{split}_mae']:>14,.2f} "
              f"{all_metrics[f'{split}_r2']:>8.3f} "
              f"{all_metrics[f'{split}_smape']:>8.2f}")

    # ── Feature importances ─────────────────────────────────────
    importances = pd.Series(final_xgb.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)
    print(f"\n  Feature importances ({len(importances)} features, sorted):")
    for feat, imp in importances.items():
        bar = "█" * int(imp * 50)
        print(f"    {feat:<22} {imp:>6.3f}  {bar}")

    # ── Tree diagnostics (XGBoost: n_estimators is depth-of-ensemble) ──
    print(f"\n  Number of trees:       {final_xgb.n_estimators}")
    print(f"  Max depth per tree:    {final_xgb.max_depth}")
    print(f"  Training samples:      {len(X_train_full)}")

    # ── Loader sanity check ─────────────────────────────────────
    try:
        loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/{artifact_name}")
        _ = loaded.predict(X_test.iloc[:1])
        print(f"\n  [OK] Model reloads and predicts")
    except Exception as e:
        print(f"\n  [WARN] Model reload failed: {e}")

🏃 View run GridSearch_XGBoost_Sunflower seed at: http://localhost:5000/#/experiments/1/runs/b79c37cacf7842dbafa124e767951730
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/15 08:14:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/15 08:14:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  [WARN] skops log failed (The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'].); retrying with cloudpickle


2026/11/15 08:14:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  XGBoost_Sunflower seed   run_id = 4c0c2cd533c445c1925343bec7b3a18b
  Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 50, 'subsample': 0.8}
  CV RMSE (5-fold on train): 252.62

  split     n           rmse            mae       r2    smape
  ------ ---- -------------- -------------- -------- --------
  train    25       7,540.99       7,441.52 -642.487   137.31
  val       4       9,231.59       9,205.95 -23378.423   148.25
  test      5       9,256.96       9,247.88 -24664.906   152.67

  Feature importances (7 features, sorted):
    precip_total            0.165  ████████
    rh2m_mean               0.162  ████████
    solar_mean              0.157  ███████
    t2m_max_extreme         0.142  ███████
    t2m_min_extreme         0.132  ██████
    ws2m_mean               0.125  ██████
    t2m_mean                0.117  █████

  Number of trees:       50
  Max depth per tree:    2
  Training samples:      436



  [OK] Model reloads and predicts
🏃 View run XGBoost_Sunflower seed at: http://localhost:5000/#/experiments/1/runs/4c0c2cd533c445c1925343bec7b3a18b
🧪 View experiment at: http://localhost:5000/#/experiments/1
